# Testing the Initial Qiskit Port

## Objective

Before attempting to convert the full variational Lanczos algorithm, the first goal is to verify that the fundamental linear algebra operations used throughout the algorithm produce identical results to the original Qulacs implementation.

Rather than immediately testing on the Anderson Impurity Model Hamiltonian, simple one-qubit examples are used where the expected results are known analytically.

---

# Step 1 – Verify the New Qiskit Module

The first step was to ensure that the newly created `qiskit_vqe.py` module was correctly recognized by Python.

Actions performed:

- Imported the helper functions:
  - `expectation_value()`
  - `transition_amplitude()`
  - `overlap()`
- Verified the module path using to ensure Python was importing the correct file. (Cell block [2])

to verify that all newly added functions were present.

Because Jupyter caches imported modules, the module was reloaded (Cell block [4]) before testing newly added functions.

This confirmed that the notebook was using the latest version of `qiskit_vqe.py`.

In [1]:
from qiskit_port.qiskit_vqe import expectation_value, transition_amplitude, overlap

print("qiskit_vqe helpers imported")

qiskit_vqe helpers imported


In [2]:
import qiskit_port.qiskit_vqe as qv
print(qv.__file__)
print(dir(qv))

/Users/aporter2/Downloads/nlr_qc/qcaob-aim/qiskit_port/qiskit_vqe.py
['__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'expectation_value', 'lanczos_cost_function_qiskit', 'np', 'overlap', 'transition_amplitude']


In [3]:
import importlib
import qiskit_port.qiskit_vqe as qv

importlib.reload(qv)

print("lanczos_cost_function_qiskit" in dir(qv))

True


In [4]:
import importlib
import qiskit_port.qiskit_vqe as qv
importlib.reload(qv)

from qiskit_port.qiskit_vqe import lanczos_cost_function_qiskit
print("import works")

import works


# Step 2 – Test the Helper Functions

Three helper functions were implemented:

- `expectation_value()`
- `transition_amplitude()`
- `overlap()`

These replace the equivalent Qulacs functionality:

| Qulacs | Qiskit/NumPy |
|---------|--------------|
| get_expectation_value() | expectation_value() |
| get_transition_amplitude() | transition_amplitude() |
| inner_product() | overlap() |

These helper routines form the mathematical foundation for the remaining variational Lanczos implementation.

In [10]:
import numpy as np

from qulacs import QuantumState
from qulacs.quantum_operator import create_quantum_operator_from_openfermion_text

from qiskit_port.qiskit_vqe import expectation_value

# Same Hamiltonian in both frameworks
qulacs_H = create_quantum_operator_from_openfermion_text("1.0 [Z0]")
Hmat = np.array([[1, 0],
                 [0, -1]], dtype=complex)

for basis in [0, 1]:

    # Qulacs state
    qs = QuantumState(1)
    qs.set_computational_basis(basis)

    # NumPy state
    state = np.zeros(2, dtype=complex)
    state[basis] = 1

    qulacs_result = qulacs_H.get_expectation_value(qs)
    qiskit_result = expectation_value(state, Hmat)

    print(f"|{basis}>")
    print("Qulacs :", qulacs_result)
    print("Qiskit :", qiskit_result)
    print("-"*40)

|0>
Qulacs : (1+0j)
Qiskit : (1+0j)
----------------------------------------
|1>
Qulacs : (-1+0j)
Qiskit : (-1+0j)
----------------------------------------


## Validation Check: Expectation Value

This cell checks that the numerical difference between the Qulacs result and the Qiskit/NumPy result is below the chosen tolerance.

If the assertion passes, then `expectation_value()` is validated for this test case.

In [12]:
tolerance = 1e-12

assert abs(qulacs_result - qiskit_result) < tolerance

print("expectation_value() matches Qulacs.")

expectation_value() matches Qulacs.


## Qulacs vs Qiskit Test: Inner Product / Overlap

This test compares the original Qulacs inner-product calculation with the new Qiskit/NumPy `overlap()` helper function.

Original Qulacs operation:

`inner_product(qs0, qs1)`

Qiskit replacement:

`overlap(state0, state1)`

The goal is to confirm that Qiskit/NumPy computes overlaps consistently with Qulacs.

In [19]:
from qulacs.state import inner_product
from qiskit_port.qiskit_vqe import overlap

qs0 = QuantumState(1)
qs0.set_computational_basis(0)

qs1 = QuantumState(1)
qs1.set_computational_basis(1)

state0 = np.array([1, 0], dtype=complex)
state1 = np.array([0, 1], dtype=complex)

qulacs_result = inner_product(qs0, qs1)
qiskit_result = overlap(state0, state1)

print("Qulacs overlap:", qulacs_result)
print("Qiskit overlap:", qiskit_result)

Qulacs overlap: 0j
Qiskit overlap: 0j


## Qulacs vs Qiskit Test: Transition Amplitude

This test compares the original Qulacs transition-amplitude calculation with the new Qiskit/NumPy `transition_amplitude()` helper function.

Original Qulacs operation:

`qulacs_H.get_transition_amplitude(qs0, qs1)`

Qiskit replacement:

`transition_amplitude(state0, Hmat, state1)`

This is important because the variational Lanczos cost function depends directly on transition amplitudes between Krylov vectors.

In [15]:
from qiskit_port.qiskit_vqe import transition_amplitude

# Hamiltonian H = X0
qulacs_H = create_quantum_operator_from_openfermion_text("1.0 [X0]")
Hmat = np.array([[0, 1], [1, 0]], dtype=complex)

qs0 = QuantumState(1)
qs0.set_computational_basis(0)

qs1 = QuantumState(1)
qs1.set_computational_basis(1)

state0 = np.array([1, 0], dtype=complex)
state1 = np.array([0, 1], dtype=complex)

# Qulacs equivalent of <0|H|1>
qulacs_result = qulacs_H.get_transition_amplitude(qs0, qs1)
qiskit_result = transition_amplitude(state0, Hmat, state1)

print("Qulacs transition:", qulacs_result)
print("Qiskit transition:", qiskit_result)

Qulacs transition: (1+0j)
Qiskit transition: (1+0j)
Difference: 0.0


## Next Validation Target: Lanczos Cost Function

The helper functions have now been tested against their Qulacs counterparts:

- `expectation_value()`
- `overlap()`
- `transition_amplitude()`

The next function to validate is:

`lanczos_cost_function_qiskit()`

This comparison requires the actual repository ansatz and AIM graph structure, because the original Qulacs function builds the trial Krylov state using `SymmQulacsVqeEmulator`.

Next step:

Use a small real AIM system, such as `N = 2`, and compare:

`vqe.lanczos_cost_function()`

against

`qiskit_port.qiskit_vqe.lanczos_cost_function_qiskit()`

using the same Hamiltonian, same Krylov states, same parameters, and same ansatz structure.

In [16]:
from qiskit import QuantumCircuit

def simple_ansatz_builder(params, n_layers, initial_occupations_indices, connected_graphs):
    qc = QuantumCircuit(1)

    for q in initial_occupations_indices:
        qc.x(q)

    qc.ry(params[0], 0)

    return qc

## Function Equivalence Test: continued_fraction()

This test compares the original repository function `vqe.continued_fraction()` against the Qiskit-port version `continued_fraction_qiskit()`.

The same frequency array `w` and Lanczos coefficient arrays `a` and `b` are passed into both functions.

This validates that the continued-fraction recursion was ported correctly.

This test does not yet validate the generation of Lanczos coefficients. We still need to implement the anderson_impurity_model.py


In [31]:
import numpy as np
import importlib

import vqe
import qiskit_port.qiskit_vqe as qv

importlib.reload(qv)

w = np.linspace(-5, 5, 20, dtype=np.complex128) + 0.1j
a = np.array([1.0, 2.0, 3.0], dtype=np.complex128)
b = np.array([0.0, 0.5, 0.25], dtype=np.complex128)

original = vqe.continued_fraction(w, a, b)
converted = qv.continued_fraction_qiskit(w, a, b)

diff = np.max(np.abs(original - converted))


assert np.allclose(original, converted)

print("continued_fraction_qiskit() matches original vqe.continued_fraction()")

continued_fraction_qiskit() matches original vqe.continued_fraction()


## Function Equivalence Test: qulacs_to_python_ordering()

This test compares the original repository function `vqe.qulacs_to_python_ordering()` against the Qiskit-port version.

This function reverses bit ordering between Qulacs and Python/Qiskit-style statevector indexing.

The test uses a simple indexed vector so the reordering is easy to inspect.

In [62]:
import numpy as np
import importlib

import vqe
import qiskit_port.qiskit_vqe as qv

importlib.reload(qv)

n_qubits = 5
state = np.arange(2**n_qubits, dtype=complex)

original = vqe.qulacs_to_python_ordering(state, n_qubits)
converted = qv.qulacs_to_python_ordering_qiskit(state, n_qubits)

print("Original:", original)
print("Converted:", converted)

assert np.allclose(original, converted)
print("qulacs_to_python_ordering_qiskit() matches original.")

Original: [ 0.+0.j 16.+0.j  8.+0.j 24.+0.j  4.+0.j 20.+0.j 12.+0.j 28.+0.j  2.+0.j
 18.+0.j 10.+0.j 26.+0.j  6.+0.j 22.+0.j 14.+0.j 30.+0.j  1.+0.j 17.+0.j
  9.+0.j 25.+0.j  5.+0.j 21.+0.j 13.+0.j 29.+0.j  3.+0.j 19.+0.j 11.+0.j
 27.+0.j  7.+0.j 23.+0.j 15.+0.j 31.+0.j]
Converted: [ 0.+0.j 16.+0.j  8.+0.j 24.+0.j  4.+0.j 20.+0.j 12.+0.j 28.+0.j  2.+0.j
 18.+0.j 10.+0.j 26.+0.j  6.+0.j 22.+0.j 14.+0.j 30.+0.j  1.+0.j 17.+0.j
  9.+0.j 25.+0.j  5.+0.j 21.+0.j 13.+0.j 29.+0.j  3.+0.j 19.+0.j 11.+0.j
 27.+0.j  7.+0.j 23.+0.j 15.+0.j 31.+0.j]
qulacs_to_python_ordering_qiskit() matches original.


## Function Equivalence Test: create_qulacs_hamiltonian()

This test compares the original repository function:

`vqe.create_qulacs_hamiltonian()`

against the Qiskit-port function:

`create_qiskit_hamiltonian_matrix()`

The original repo function creates Qulacs Hamiltonian objects:

- shifted Hamiltonian \(H - E_0 I\)
- squared shifted Hamiltonian \((H - E_0 I)^2\)
- number of qubits

The Qiskit-port version creates the equivalent NumPy matrices using OpenFermion.

The goal is to verify that both versions represent the same Hamiltonian.

### Initial Result

The first direct comparison failed.

This does not necessarily mean the Hamiltonian conversion is wrong. The likely issue is basis ordering / endianness between Qulacs and the NumPy/Qiskit matrix representation.

### Next Diagnostic Step

To isolate the problem, I printed the first computational basis state where the Qulacs expectation value and the Qiskit/NumPy expectation value disagreed.

Then I repeated the test after applying the ordering conversion:

`qulacs_to_python_ordering_qiskit()`

This checks whether the mismatch comes from statevector ordering rather than the Hamiltonian itself.

### Interpretation

If the comparison passes after applying the ordering conversion, then the Hamiltonian construction is consistent and the mismatch was caused by basis ordering.

If it still fails, then the Hamiltonian matrix construction needs further debugging.

In [36]:
for basis in range(2**n_qubits_qulacs):
    qs = QuantumState(n_qubits_qulacs)
    qs.set_computational_basis(basis)

    state = np.zeros(2**n_qubits_qiskit, dtype=complex)
    state[basis] = 1.0

    qulacs_exp_H = qulacs_H.get_expectation_value(qs)
    qiskit_exp_H = np.vdot(state, Hmat @ state)

    diff = abs(qulacs_exp_H - qiskit_exp_H)

    if diff > 1e-10:
        print("basis:", basis)
        print("Qulacs:", qulacs_exp_H)
        print("Qiskit:", qiskit_exp_H)
        print("diff:", diff)
        print("-" * 30)
        break

basis: 1
Qulacs: (3.5795440294030243+0j)
Qiskit: (-1.4108324970703672+0j)
diff: 4.9903765264733915
------------------------------


## Validation Results

To further verify the Hamiltonian conversion, I computed the expectation values of both the shifted Hamiltonian \(H\) and the squared Hamiltonian \(H^2\) for every computational basis state.

For each basis state, I evaluated the expectation values using:

- the original **Qulacs** Hamiltonian objects, and
- the converted **Qiskit/OpenFermion** Hamiltonian matrices.

Before evaluating the Qiskit Hamiltonian, the basis state was converted using `qulacs_to_python_ordering_qiskit()` to account for the difference in statevector ordering between Qulacs and Qiskit.

During the validation, I recorded:

- the maximum expectation value of \(H\) from both implementations,
- the maximum expectation value of \(H^2\) from both implementations,
- the maximum absolute difference between the expectation values for \(H\), and
- the maximum absolute difference between the expectation values for \(H^2\).

The observed maximum differences were

- **Max \(H\) expectation difference:** `4.440892098500626e-16`
- **Max \(H^2\) expectation difference:** `2.842170943040401e-14`

These values are at the level of floating-point precision, indicating that the Qiskit/OpenFermion Hamiltonian reproduces the original Qulacs Hamiltonian after correcting for basis ordering. This confirms that the Hamiltonian conversion is numerically equivalent to the original implementation.

In [49]:
from qiskit_port.qiskit_vqe import qulacs_to_python_ordering_qiskit
from qulacs import QuantumState
import numpy as np

max_diff_H = 0.0
max_diff_H2 = 0.0

max_qulacs_exp_H = -np.inf
max_qiskit_exp_H = -np.inf

max_qulacs_exp_H2 = -np.inf
max_qiskit_exp_H2 = -np.inf

for basis in range(2**n_qubits_qulacs):
    # Create the Qulacs computational basis state
    qs = QuantumState(n_qubits_qulacs)
    qs.set_computational_basis(basis)

    # Equivalent NumPy state
    qulacs_state = np.zeros(2**n_qubits_qulacs, dtype=complex)
    qulacs_state[basis] = 1.0

    # Convert from Qulacs ordering to Python/Qiskit ordering
    state = qulacs_to_python_ordering_qiskit(qulacs_state, n_qubits_qulacs)

    # Expectation values of H
    qulacs_exp_H = qulacs_H.get_expectation_value(qs).real
    qiskit_exp_H = np.vdot(state, Hmat @ state).real

    # Expectation values of H^2
    qulacs_exp_H2 = qulacs_H2.get_expectation_value(qs).real
    qiskit_exp_H2 = np.vdot(state, H2mat @ state).real

    # Track maximum expectation values
    max_qulacs_exp_H = max(max_qulacs_exp_H, qulacs_exp_H)
    max_qiskit_exp_H = max(max_qiskit_exp_H, qiskit_exp_H)

    max_qulacs_exp_H2 = max(max_qulacs_exp_H2, qulacs_exp_H2)
    max_qiskit_exp_H2 = max(max_qiskit_exp_H2, qiskit_exp_H2)

    # Track maximum absolute differences
    max_diff_H = max(max_diff_H, abs(qulacs_exp_H - qiskit_exp_H))
    max_diff_H2 = max(max_diff_H2, abs(qulacs_exp_H2 - qiskit_exp_H2))

print("===== H =====")
print("Max Qulacs expectation :", max_qulacs_exp_H)
print("Max Qiskit expectation :", max_qiskit_exp_H)
print("Max difference         :", max_diff_H)

print("\n===== H^2 =====")
print("Max Qulacs expectation :", max_qulacs_exp_H2)
print("Max Qiskit expectation :", max_qiskit_exp_H2)
print("Max difference         :", max_diff_H2)

assert max_diff_H < 1e-10
assert max_diff_H2 < 1e-10


===== H =====
Max Qulacs expectation : 14.758884722531484
Max Qiskit expectation : 14.758884722531484
Max difference         : 4.440892098500626e-16

===== H^2 =====
Max Qulacs expectation : 219.08645300731536
Max Qiskit expectation : 219.08645300731538
Max difference         : 2.842170943040401e-14


## Controlled Unit Test vs Full Workflow Test

This test is not the full AIM/VQE workflow yet. It is a controlled unit test.

The purpose is to verify that one converted function gives the same output as the original repository function when both are given the same inputs.

For example:

`vqe.get_initial_occupations_indices()`

is compared directly against

`qiskit_port.qiskit_vqe.get_initial_occupations_indices_qiskit()`

using real AIM system inputs generated by:

`dmft.initialize_system(system_size, seed)`

This means the qubit indices come from the actual repository workflow, but the test avoids running the full VQE optimizer.

This is intentional. The conversion is being validated one function at a time before integrating the full workflow.

In [54]:
import importlib
import vqe
import qiskit_port.qiskit_vqe as qv

importlib.reload(qv)

up_qubit_indices = [0, 1, 2]
down_qubit_indices = [3, 4, 5]

test_cases = [
    (1, 1),
    (2, 1),
    (1, 2),
    (2, 2),
    (3, 3),
]

for n_up, n_down in test_cases:
    original = vqe.get_initial_occupations_indices(
        up_qubit_indices,
        down_qubit_indices,
        n_up,
        n_down,
    )

    converted = qv.get_initial_occupations_indices_qiskit(
        up_qubit_indices,
        down_qubit_indices,
        n_up,
        n_down,
    )

    print(f"n_up={n_up}, n_down={n_down}")
    print("Original :", original)
    print("Converted:", converted)
    print()

    assert original == converted


n_up=1, n_down=1
Original : [0, 3]
Converted: [0, 3]

n_up=2, n_down=1
Original : [0, 2, 3]
Converted: [0, 2, 3]

n_up=1, n_down=2
Original : [0, 3, 5]
Converted: [0, 3, 5]

n_up=2, n_down=2
Original : [0, 2, 3, 5]
Converted: [0, 2, 3, 5]

n_up=3, n_down=3
Original : [0, 1, 2, 3, 4, 5]
Converted: [0, 1, 2, 3, 4, 5]



In [65]:
import numpy as np
import importlib

from qulacs import QuantumState
import anderson_impurity_model as aim
import dmft
import vqe

import qiskit_port.qiskit_ansatz as qa
from qiskit_port.qiskit_vqe import qulacs_to_python_ordering_qiskit
from qiskit.quantum_info import Statevector

importlib.reload(qa)

system_size = 3
seed = 0
vqe_depth = 1

(
    impurity_orbital,
    test_model,
    n_orbitals,
    up_qubit_indices,
    down_qubit_indices,
    connected_graphs,
    qubit_hamiltonian,
) = dmft.initialize_system(system_size, seed)

# Use repo-generated qubit count
qulacs_ham, _, n_qubits = vqe.create_qulacs_hamiltonian(
    qubit_hamiltonian,
    vqe_gs_energy=-1.0,
)

# Use a real occupation sector
n_up = 1
n_down = 1

initial_occupations_indices = vqe.get_initial_occupations_indices(
    up_qubit_indices,
    down_qubit_indices,
    n_up,
    n_down,
)

n_edges = connected_graphs["graph_full"].number_of_edges()
n_params = vqe_depth * (n_edges + n_qubits)

np.random.seed(0)
theta = np.random.uniform(0, 2 * np.pi, size=n_params)

# Original Qulacs ansatz
symm_qulacs = aim.SymmQulacsVqeEmulator(qulacs_ham)
qulacs_circuit = symm_qulacs.all_symmetry_ansatzae(
    theta=theta,
    n_layers=vqe_depth,
    initial_occupations_indices=initial_occupations_indices,
    connected_graphs=connected_graphs,
    compilation="generic",
)

qs = QuantumState(n_qubits)
qs.set_zero_state()
qulacs_circuit.update_quantum_state(qs)

qulacs_state = qs.get_vector()
qulacs_state_ordered = qulacs_to_python_ordering_qiskit(qulacs_state, n_qubits)

# Qiskit ansatz
qiskit_circuit = qa.all_symmetry_ansatzae_qiskit(
    theta=theta,
    n_qubits=n_qubits,
    n_layers=vqe_depth,
    initial_occupations_indices=initial_occupations_indices,
    connected_graphs=connected_graphs,
    compilation="generic",
)

qiskit_state = Statevector.from_instruction(qiskit_circuit).data

diff = np.max(np.abs(qulacs_state_ordered - qiskit_state))

print("n_qubits:", n_qubits)
print("n_params:", n_params)
print("initial occupations:", initial_occupations_indices)
print("max statevector difference:", diff)

assert np.allclose(qulacs_state_ordered, qiskit_state, atol=1e-10)

print("Qiskit generic ansatz matches Qulacs generic ansatz.")

n_qubits: 6
n_params: 13
initial occupations: [0, 3]
max statevector difference: 0.768550427416138


AssertionError: 